In [6]:
import sys
 
print("Python executable:")
print(sys.executable)
 
print("\nPython version:")
print(sys.version)
 
# Jalankan sekali bila tqdm belum terpasang (sintaks notebook):
# !{sys.executable} -m pip install tqdm
 
# %%
import tqdm
print(tqdm.__version__, tqdm.__file__)
 
# %%
# ============================================================
# 0. IMPOR, LOGGING, QUIET
# ============================================================
import csv
import gc
import hashlib
import os
import pickle
import time
import logging
import contextlib
import importlib
from pathlib import Path
 
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from sklearn.metrics import (
    auc as sk_auc, roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve,
)
 
from dataset import prepare_client_data, TARGET, DROP_COLS
 
# Windows: start Ray kadang lambat/flaky saat di-restart berkali-kali.
os.environ.setdefault("RAY_NODE_START_TIMEOUT", "180")
 
VERBOSE = False
OUT_DIR = Path("hasil_fl")
OUT_DIR.mkdir(exist_ok=True)
LOG_FLWR = OUT_DIR / "flwr.log"
LOG_PRINTS = OUT_DIR / "prints.log"
 
 
def setup_flwr_logging():
    """Console Flower hanya WARNING+; detail penuh ke file.
    Panggil SEBELUM tiap run (Flower bisa memasang ulang handler console)."""
    lg = logging.getLogger("flwr")
    lg.propagate = False
    for h in list(lg.handlers):
        if not isinstance(h, logging.FileHandler):
            h.setLevel(logging.WARNING)
    if not any(isinstance(h, logging.FileHandler) for h in lg.handlers):
        fh = logging.FileHandler(LOG_FLWR, encoding="utf-8")
        fh.setLevel(logging.INFO)
        fh.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        lg.addHandler(fh)
    lg.setLevel(logging.INFO)
 
 
@contextlib.contextmanager
def quiet(logfile=None):
    """Alihkan print() (stdout) ke file. Dinonaktifkan bila VERBOSE.
    tqdm menulis ke stderr, jadi progress bar tetap tampil."""
    if VERBOSE:
        yield
        return
    with open(logfile or LOG_PRINTS, "a", encoding="utf-8") as f, \
            contextlib.redirect_stdout(f):
        yield
 
 
def run_quiet(fn, *args, **kwargs):
    """Jalankan fungsi training dengan log Flower dan print() dialihkan ke file."""
    setup_flwr_logging()
    with quiet():
        out = fn(*args, **kwargs)
    setup_flwr_logging()
    return out
 
 
setup_flwr_logging()

Python executable:
c:\Users\s2\Documents\FIXRKI\.fed\Scripts\python.exe

Python version:
3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
4.70.1 c:\Users\s2\Documents\FIXRKI\.fed\Lib\site-packages\tqdm\__init__.py


In [7]:
# ============================================================
# 1. KONFIGURASI
# ============================================================
CLIENT_DIR = Path(r'C:\Users\s2\Documents\FIXRKI\bfprssprep\split\clients')
 
# --- Seed ---
SINGLE_SEED = 42                      # run tunggal (bagian 7 & 8)
SEEDS = (42, 0, 123, 1337, 3407)      # multi-seed (bagian 9)
 
# --- Threshold (diteruskan ke fl_runner) ---
#   THR_MODE : "fixed" | "per_client" | "global"
#              (per_client/global: threshold dipilih dari VAL, dipakai di TEST;
#               metrik VAL akhir memakai threshold yang dituning di VAL itu
#               sendiri -> in-sample/optimistis)
#   THR_CRIT : "f1" | "youden" | "gmean" | "sens_target"
THR_MODE = "fixed"
THR_CRIT = "f1"
THR_KW = dict(threshold_mode=THR_MODE, threshold_criterion=THR_CRIT)
 
# --- Evaluasi akhir: validasi dan/atau test ---
EVAL_SPLITS = ("val", "test")
EVAL_KW = dict(eval_splits=EVAL_SPLITS)
 
# Ubah bila logika/versi fl_runner berubah, agar cache lama tidak terpakai.
CACHE_VERSION = "v2_noadapter_valtest"
 
# ============================================================
# 2. MUAT DATA SEMUA CLIENT
# ============================================================
clients, scaler, feature_names = prepare_client_data(CLIENT_DIR)
 
print(f"Jumlah client : {len(clients)}")
print(f"Daftar state  : {list(clients.keys())}")
print(f"Target        : {TARGET}")
print(f"Kolom dibuang : {DROP_COLS}")
 
# ============================================================
# 3. DAFTAR FITUR
#    Tanpa adapter, SEMUA client harus punya jumlah fitur yang sama.
# ============================================================
n_fitur_per_client = {st: c['X_train_t'].shape[1] for st, c in clients.items()}
assert len(set(n_fitur_per_client.values())) == 1, \
    f"Jumlah fitur beda antar client (tidak boleh tanpa adapter): {n_fitur_per_client}"
n_fitur = next(iter(n_fitur_per_client.values()))
assert n_fitur == len(feature_names), "Jumlah fitur tidak cocok dengan nama fitur"
 
print(f"\nJumlah fitur  : {n_fitur}")
for i, nama in enumerate(feature_names, 1):
    print(f"{i:>3}. {nama}")

Jumlah client : 44
Daftar state  : [1, 2, 4, 5, 8, 10, 11, 12, 13, 15, 16, 17, 18, 19, 20, 22, 23, 24, 25, 26, 27, 29, 30, 31, 33, 34, 35, 36, 37, 38, 39, 40, 41, 44, 45, 46, 48, 49, 50, 51, 53, 54, 55, 56]
Target        : _michd
Kolom dibuang : ['iyear', '_state', 'cvdinfr4', 'cvdcrhd4']

Jumlah fitur  : 69
  1. sexvar
  2. _age80
  3. _educag
  4. income3
  5. veteran3
  6. children
  7. _urbstat
  8. _bmi5
  9. diabete4
 10. medcost1
 11. checkup1
 12. genhlth
 13. physhlth
 14. menthlth
 15. exerany2
 16. cvdstrk3
 17. chcscnc1
 18. chcocnc1
 19. chccopd3
 20. addepev3
 21. chckdny2
 22. havarth4
 23. diabage4
 24. deaf
 25. blind
 26. decide
 27. diffwalk
 28. diffdres
 29. diffalon
 30. _rfbing6
 31. drnkany6
 32. flushot7
 33. pneuvac4
 34. marital_1
 35. marital_2
 36. marital_3
 37. marital_4
 38. marital_5
 39. marital_6
 40. employ1_1
 41. employ1_2
 42. employ1_3
 43. employ1_4
 44. employ1_5
 45. employ1_6
 46. employ1_7
 47. employ1_8
 48. renthom1_1
 49. renthom1_2
 50. 

In [8]:
# ============================================================
# 4. RINGKASAN TIAP CLIENT
# ============================================================
rows = {}
for st, c in clients.items():
    rows[st] = {
        'n_train': len(c['y_train_t']),
        'n_val': len(c['y_val_t']),
        'n_test': len(c['y_test_t']),
        'pos_train': int(c['y_train_t'].sum()),
        'pos_val': int(c['y_val_t'].sum()),
        'pos_test': int(c['y_test_t'].sum()),
    }
 
ringkas = pd.DataFrame(rows).T
ringkas.index.name = 'state'
 
for s in ('train', 'val', 'test'):
    ringkas[f'pos_{s}_%'] = (100 * ringkas[f'pos_{s}'] / ringkas[f'n_{s}']).round(2)
 
total = ringkas[[c for c in ringkas.columns if not c.endswith('%')]].sum()
for s in ('train', 'val', 'test'):
    total[f'pos_{s}_%'] = round(100 * total[f'pos_{s}'] / total[f'n_{s}'], 2)
ringkas.loc['TOTAL'] = total
 
print("\nRingkasan per client:")
print(ringkas.to_string())


Ringkasan per client:
       n_train    n_val   n_test  pos_train  pos_val  pos_test  pos_train_%  pos_val_%  pos_test_%
state                                                                                             
1        979.0    562.0    581.0      236.0    124.0     131.0        24.11      22.06       22.55
2        644.0    358.0    389.0      126.0     67.0      84.0        19.57      18.72       21.59
4       1781.0    804.0    532.0      391.0    170.0     109.0        21.95      21.14       20.49
5       1076.0    595.0    544.0      279.0    154.0     147.0        25.93      25.88       27.02
8        927.0    646.0    601.0      151.0    133.0      99.0        16.29      20.59       16.47
10       686.0    414.0    390.0      144.0     96.0      87.0        20.99      23.19       22.31
11       345.0    199.0    200.0       49.0     29.0      25.0        14.20      14.57       12.50
12      2330.0   1257.0   1276.0      589.0    274.0     299.0        25.28      21.80

In [9]:

# %%
# ============================================================
# 5. clients -> fed_data
# ============================================================
fed_data = {}
for st, c in clients.items():
    fed_data[f"state_{st}"] = {
        "X_train": c["X_train_t"], "y_train": c["y_train_t"],
        "X_val": c["X_val_t"], "y_val": c["y_val_t"],
        "X_test": c["X_test_t"], "y_test": c["y_test_t"],
        "n_features": c["X_train_t"].shape[1],
    }
 
print("Client:", list(fed_data.keys()))
print("n_features per client:", {k: v["n_features"] for k, v in fed_data.items()})
 
# %%
# ============================================================
# 6. IMPOR RUNNER  (file: fl_running1.py = fl_runner versi terbaru)
# ============================================================
import fl_running1
importlib.reload(fl_running1)
from fl_running1 import (
    METRIC_KEYS, POOLED, eval_prefix,
    run_federated, report_results,
    print_comparison, print_val_test_comparison,
    export_history_csv, export_per_client_csv,
    export_comparison_csv, export_rounds_comparison_csv,
    export_curves_csv, export_threshold_sweep_csv,
    export_val_test_summary_csv,
    plot_diagnostic_curves, plot_comparison_curves, plot_val_test_curves,
)
 
N_CLASSES = 2          # (COMMON_DIM dihapus: tidak ada adapter)
 
# %%
# ============================================================
# 6a. HELPER PROGRESS (tqdm) + RINGKASAN ANTAR SEED
# ============================================================
def round_bar(total, desc, position=0, leave=True):
    """Progress bar per ronde + callback `on_round` untuk run_federated."""
    bar = tqdm(total=total, desc=desc, position=position, leave=leave,
               unit="ronde", dynamic_ncols=True)
 
    def on_round(rnd, n_rounds, loss, m):
        bar.update(1)
        bar.set_postfix(loss=f"{loss:.3f}", f1=f"{m['f1']:.3f}",
                        auc=f"{m['auc']:.3f}", ap=f"{m['ap']:.3f}")
 
    return bar, on_round
 
 
def _reset_ray(wait=8):
    """Matikan sesi Ray yang menggantung, lalu beri jeda agar port/proses bebas."""
    try:
        import ray
        if ray.is_initialized():
            ray.shutdown()
    except Exception:
        pass
    gc.collect()
    time.sleep(wait)
 
 
RUN_CACHE = OUT_DIR / "runs"
RUN_CACHE.mkdir(exist_ok=True)
RESUME = True      # True: lewati run yang sudah selesai (hasil_fl/runs/*.pkl)
 
 
def _cache_path(tag, kw):
    items = sorted((k, repr(v)) for k, v in kw.items()
                   if k not in ("on_round", "fed_data"))
    key = hashlib.md5(repr((CACHE_VERSION, items)).encode()).hexdigest()[:10]
    return RUN_CACHE / f"{tag}_{key}.pkl"
 
 
def _run_complete(h, kw):
    """Run dianggap lengkap bila semua ronde selesai & evaluasi akhir ada."""
    if len(h["round"]) < kw["num_rounds"]:
        return False
    splits = kw.get("eval_splits", EVAL_SPLITS)
    return all(h.get(eval_prefix(s, True)) for s in splits)
 
 
def run_one(tag, bar, retries=3, **kw):
    """Satu run_federated dengan (a) cache hasil, (b) retry bila Ray gagal start
    atau simulasi tidak lengkap (mis. TimeoutError node Ray di Windows)."""
    cp = _cache_path(tag, kw)
    if RESUME and cp.exists():
        with open(cp, "rb") as f:
            h = pickle.load(f)
        bar.update(kw["num_rounds"] - bar.n)
        return h
 
    last = None
    for attempt in range(1, retries + 1):
        bar.reset()
        try:
            h = run_quiet(run_federated, **kw)
            if _run_complete(h, kw):
                with open(cp, "wb") as f:
                    pickle.dump(h, f)
                return h
            last = RuntimeError(f"simulasi tidak lengkap "
                                f"({len(h['round'])}/{kw['num_rounds']} ronde)")
        except Exception as e:                      # noqa: BLE001
            last = e
        tqdm.write(f"[retry {attempt}/{retries}] {tag} gagal: "
                   f"{type(last).__name__}: {str(last)[:120]}")
        _reset_ray(wait=8 * attempt)
    raise RuntimeError(f"{tag} gagal setelah {retries} percobaan") from last
 
 
def run_grid(fed_data, modes, aggs, seeds, num_rounds, **run_kwargs):
    """Semua kombinasi mode x agregasi x seed dengan progress bar dua tingkat.
    all_hist[mode][agg] = history seed pertama + ["runs"]."""
    total = len(modes) * len(aggs) * len(seeds)
    outer = tqdm(total=total, desc="Total run", position=0, unit="run",
                 dynamic_ncols=True)
    all_hist = {}
    try:
        for mode in modes:
            all_hist[mode] = {}
            for agg in aggs:
                runs = []
                for s in seeds:
                    outer.set_postfix_str(f"{mode}+{agg} seed={s}")
                    bar, cb = round_bar(num_rounds, f"  {mode}+{agg} s={s}",
                                        position=1, leave=False)
                    try:
                        h = run_one(
                            f"{mode}_{agg}_s{s}", bar,
                            fed_data=fed_data, name="cmp",
                            aggregation=agg, seed=s, mode=mode,
                            num_rounds=num_rounds, on_round=cb,
                            verbose_report=False, **run_kwargs)
                    finally:
                        bar.close()
                    runs.append(h)
                    outer.update(1)
                h0 = dict(runs[0])          # salinan dangkal
                h0["runs"] = runs
                all_hist[mode][agg] = h0
    finally:
        outer.close()
    return all_hist
 
 
def seed_summary(histories, split="test", ddof=1):
    """{agg: {metrik: (mean, std)}} antar seed.
    split: 'test' | 'test_best' | 'val_last' | 'val_best'."""
    out = {}
    for agg, h in histories.items():
        runs = [r for r in h.get("runs", [h]) if r.get(split)]
        out[agg] = {}
        for k in METRIC_KEYS:
            v = np.array([r[split][k] for r in runs], dtype=float)
            v = v[~np.isnan(v)]
            if v.size == 0:
                out[agg][k] = (float("nan"), float("nan"))
            else:
                sd = float(v.std(ddof=ddof)) if v.size > ddof else float("nan")
                out[agg][k] = (float(v.mean()), sd)
    return out
 
 
def print_seed_summary(histories, split="test", ddof=1):
    summ = seed_summary(histories, split, ddof)
    tbl = {agg: {k: f"{mu:.4f} ± {sd:.4f}" for k, (mu, sd) in d.items()}
           for agg, d in summ.items()}
    print(f"\n=== Ringkasan antar seed ({split}, ddof={ddof}) ===")
    print(pd.DataFrame(tbl).T.to_string())
 
 
def export_seed_summary_csv(histories, out_path, split="test_best", ddof=1):
    summ = seed_summary(histories, split, ddof)
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["aggregation", *[f"{k}_{s}" for k in METRIC_KEYS
                                      for s in ("mean", "std")]])
        for agg, d in summ.items():
            w.writerow([agg, *[v for k in METRIC_KEYS for v in d[k]]])
    print(f"[export] ringkasan antar seed tersimpan di: {out_path}")
    return out_path
 
 
def export_seed_runs_csv(histories, out_path, split="test_best"):
    with open(out_path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["aggregation", "seed", "best_round", *METRIC_KEYS])
        for agg, h in histories.items():
            for r in h.get("runs", [h]):
                if r.get(split):
                    w.writerow([agg, r["seed"], r.get("best_round"),
                                *[r[split][k] for k in METRIC_KEYS]])
    print(f"[export] hasil tiap seed tersimpan di: {out_path}")
    return out_path

Client: ['state_1', 'state_2', 'state_4', 'state_5', 'state_8', 'state_10', 'state_11', 'state_12', 'state_13', 'state_15', 'state_16', 'state_17', 'state_18', 'state_19', 'state_20', 'state_22', 'state_23', 'state_24', 'state_25', 'state_26', 'state_27', 'state_29', 'state_30', 'state_31', 'state_33', 'state_34', 'state_35', 'state_36', 'state_37', 'state_38', 'state_39', 'state_40', 'state_41', 'state_44', 'state_45', 'state_46', 'state_48', 'state_49', 'state_50', 'state_51', 'state_53', 'state_54', 'state_55', 'state_56']
n_features per client: {'state_1': 69, 'state_2': 69, 'state_4': 69, 'state_5': 69, 'state_8': 69, 'state_10': 69, 'state_11': 69, 'state_12': 69, 'state_13': 69, 'state_15': 69, 'state_16': 69, 'state_17': 69, 'state_18': 69, 'state_19': 69, 'state_20': 69, 'state_22': 69, 'state_23': 69, 'state_24': 69, 'state_25': 69, 'state_26': 69, 'state_27': 69, 'state_29': 69, 'state_30': 69, 'state_31': 69, 'state_33': 69, 'state_34': 69, 'state_35': 69, 'state_36': 69, '

In [10]:

# %%
# ============================================================
# 6b. HELPER DIAGNOSTIK (AUC-ROC, AP, kurva ROC/PR) -- val & test
#     Semua sudah dihitung di fl_runner (history[<prefix>_per_client],
#     history[<prefix>_curves]); tidak perlu training ulang.
#     prefix: 'test_best' | 'test' | 'val_best' | 'val_last'
#     Kurva rata-rata = macro-average antar client (POOLED dilewati).
# ============================================================
GRID = np.linspace(0, 1, 201)
 
 
def _safe(fn, y, p):
    """NaN bila hanya satu kelas."""
    return float(fn(y, p)) if 0 < y.sum() < len(y) else float("nan")
 
 
def diagnostic_metrics(ys, ps):
    """AUC-ROC, AP, PR-AUC (trapesium) gabungan + per state + titik kurva.
    Dipakai oleh collect_result (bagian 13) yang punya probabilitas mentah."""
    y_all = np.concatenate(list(ys.values()))
    p_all = np.concatenate(list(ps.values()))
    per_state = pd.DataFrame({
        st: {"auc": _safe(roc_auc_score, ys[st], ps[st]),
             "ap": _safe(average_precision_score, ys[st], ps[st]),
             "prevalensi": float(ys[st].mean())}
        for st in ys}).T
    per_state.index.name = "state"
    fpr, tpr, _ = roc_curve(y_all, p_all)
    prec, rec, _ = precision_recall_curve(y_all, p_all)
    return {
        "auc": _safe(roc_auc_score, y_all, p_all),
        "ap": _safe(average_precision_score, y_all, p_all),
        "prauc": float(sk_auc(rec, prec)) if len(rec) > 1 else float("nan"),
        "prevalensi": float(y_all.mean()),
        "per_state": per_state,
        "roc": (fpr, tpr),
        "pr": (rec, prec),
    }
 
 
def macro_curve(curves, kind):
    """Rata-rata kurva antar client pada grid seragam.
    curves : history["<prefix>_curves"]; kind: 'roc' | 'pr'."""
    ys = []
    for name, c in (curves or {}).items():
        if name == POOLED:
            continue
        if kind == "roc" and c.get("fpr"):
            ys.append(np.interp(GRID, c["fpr"], c["tpr"]))
        elif kind == "pr" and c.get("recall"):
            rec = np.asarray(c["recall"])[::-1]      # recall menurun -> naik
            prec = np.asarray(c["precision"])[::-1]
            ys.append(np.interp(GRID, rec, prec))
    return np.mean(ys, axis=0) if ys else None
 
 
def collect_seed_diagnostics(all_hist, prefix="test_best"):
    """Kumpulkan (seed-level, per-state, kurva) dari all_hist untuk satu prefix."""
    split_pc, split_curves = f"{prefix}_per_client", f"{prefix}_curves"
    seed_rows, state_rows, curves = [], [], {}
    for mode, hs in all_hist.items():
        for agg, h in hs.items():
            for r in h.get("runs", [h]):
                m, pc = r.get(prefix), r.get(split_pc)
                if not m or not pc:
                    continue
                prev = float(np.average(
                    [v.get("prevalence", np.nan) for v in pc.values()],
                    weights=[v["n"] for v in pc.values()]))
                seed_rows.append({
                    "mode": mode, "agg": agg, "seed": r["seed"],
                    "best_round": r.get("best_round"),
                    **{k: m[k] for k in METRIC_KEYS}, "prevalensi": prev})
                for cn, v in pc.items():
                    state_rows.append({
                        "mode": mode, "agg": agg, "seed": r["seed"], "state": cn,
                        "auc": v["auc"], "ap": v["ap"],
                        "prevalensi": v.get("prevalence", np.nan)})
                cv = r.get(split_curves)
                roc, pr = macro_curve(cv, "roc"), macro_curve(cv, "pr")
                if roc is not None and pr is not None:
                    curves.setdefault((mode, agg), []).append((roc, pr))
    return pd.DataFrame(seed_rows), pd.DataFrame(state_rows), curves
 
 
def plot_seed_curves(curves, seed_df, out_path, title_tail=""):
    """ROC & PR rata-rata antar seed (± std) untuk tiap mode+agregasi."""
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
    for (mode, agg), lst in curves.items():
        tprs = np.array([c[0] for c in lst])
        precs = np.array([c[1] for c in lst])
        lab = f"{mode}+{agg}"
        t_m, t_s = tprs.mean(0), tprs.std(0)
        p_m, p_s = precs.mean(0), precs.std(0)
        a1.plot(GRID, t_m, label=lab)
        a1.fill_between(GRID, t_m - t_s, t_m + t_s, alpha=.15)
        a2.plot(GRID, p_m, label=lab)
        a2.fill_between(GRID, p_m - p_s, p_m + p_s, alpha=.15)
    a1.plot([0, 1], [0, 1], "k--", alpha=.5)
    a1.set(title=f"ROC {title_tail}", xlabel="FPR", ylabel="TPR")
    base = seed_df["prevalensi"].mean()
    a2.axhline(base, color="k", ls="--", alpha=.5, label=f"acak (={base:.3f})")
    a2.set(title=f"Precision-Recall {title_tail}", xlabel="Recall",
           ylabel="Precision")
    for a in (a1, a2):
        a.grid(alpha=.3)
        a.legend(fontsize=7)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150)
    plt.show()
 
# %%
# ============================================================
# 7. TES CEPAT (sanity check)
# ============================================================
bar, cb = round_bar(2, "smoke")
try:
    h_test = run_quiet(
        run_federated,
        fed_data, name="smoke",
        n_classes=N_CLASSES,
        num_rounds=2, local_epochs=1,
        mode="sim", aggregation="fedavg",
        seed=SINGLE_SEED,
        verbose_report=False, on_round=cb, **THR_KW, **EVAL_KW,
    )
finally:
    bar.close()
print("val f1/ap per ronde :", h_test["f1"], h_test["ap"])
print("VAL  ronde terbaik  :", h_test.get("val_best"))
print("TEST ronde terbaik  :", h_test.get("test_best"))
print("comm                :", h_test["comm"])
 
# ============================================================
# 8. SATU KONFIGURASI PENUH
# ============================================================
FULL_ROUNDS = 10
bar, cb = round_bar(FULL_ROUNDS, "exp alt+pairwise")
try:
    history = run_quiet(
        run_federated,
        fed_data, name="exp",
        n_classes=N_CLASSES,
        num_rounds=FULL_ROUNDS, local_epochs=10, batch_size=64, lr=1e-3,
        mode="alt", aggregation="pairwise",
        epochs_v=3, epochs_u=2,
        select_by="f1",
        seed=SINGLE_SEED,
        verbose_report=False, on_round=cb, **THR_KW, **EVAL_KW,
    )
finally:
    bar.close()
 
# laporan lengkap (val + test + perbandingan) dicetak ke layar
history["summary"] = report_results(history, client_label="state")
 
print("\nVAL  ronde terbaik (rata-rata client):", history.get("val_best"))
print("TEST ronde terbaik (rata-rata client):", history.get("test_best"),
      "| best_round =", history.get("best_round"))
 
# kurva ROC/PR per client + pooled, serta val vs test
plot_diagnostic_curves(history, str(OUT_DIR / "roc_pr_single_test.png"),
                       split="test_best_curves",
                       title="alt+pairwise (TEST, ronde terbaik)")
plot_diagnostic_curves(history, str(OUT_DIR / "roc_pr_single_val.png"),
                       split="val_best_curves",
                       title="alt+pairwise (VAL, ronde terbaik)")
plot_val_test_curves(history, str(OUT_DIR / "roc_pr_single_val_vs_test.png"),
                     best=True)
 
# %%
# ============================================================
# 9. BANDINGKAN mode x aggregation (multi-seed)
#    Default 5 seed x 3 agregasi x 2 mode = 30 run.
#    Print dialihkan ke hasil_fl/prints.log (VERBOSE=True untuk layar).
# ============================================================
NUM_ROUNDS = 10
LOCAL_EPOCHS = 10
AGGS = ("fedavg", "fedprox", "pairwise")
MODES = ("sim", "alt")
 
all_hist = run_grid(
    fed_data, MODES, AGGS, SEEDS, NUM_ROUNDS,
    n_classes=N_CLASSES, local_epochs=LOCAL_EPOCHS,
    fedprox_mu=0.01, select_by="f1", **THR_KW, **EVAL_KW,
)
 
 
def _fmt(ms):
    mu, sd = ms
    return f"{mu:.4f} ± {sd:.4f}"
 
 
rows = []
for mode, hs in all_hist.items():
    for agg, h in hs.items():
        runs = h.get("runs", [h])
        st = seed_summary({agg: h}, "test", ddof=1)[agg]
        sb = seed_summary({agg: h}, "test_best", ddof=1)[agg]
        vb = seed_summary({agg: h}, "val_best", ddof=1)[agg]
        row = {
            "config": f"{mode}+{agg}",
            "n_seeds": len(runs),
            "val_f1_last_round": np.nanmean([r["f1"][-1] for r in runs if r["f1"]]),
            "best_round": np.nanmean([r.get("best_round") for r in runs
                                      if r.get("best_round") is not None]),
            "total_MB": np.mean([r["comm"]["total_mb"] for r in runs]),
        }
        for k in ("f1", "auc", "ap"):
            row[f"val_best_{k}"] = _fmt(vb[k])
        for k in ("f1", "auc", "ap"):
            row[f"test_last_{k}"] = _fmt(st[k])
        for k in METRIC_KEYS:
            row[f"test_best_{k}"] = _fmt(sb[k])
        # gap val - test (F1, ronde terbaik): indikasi overfit ke validasi
        row["gap_f1_val_test"] = vb["f1"][0] - sb["f1"][0]
        rows.append(row)
summary = pd.DataFrame(rows).set_index("config")
print("\n", summary.round(4).to_string())
summary.to_csv(OUT_DIR / "summary.csv")
 
# ============================================================
# 9b. DIAGNOSTIK: AUC-ROC, AP, kurva ROC & PR (mean ± std antar seed)
#     Untuk VAL (ronde terbaik) dan TEST (ronde terbaik).
# ============================================================
diag_frames = {}
for split_name, prefix in (("val", "val_best"), ("test", "test_best")):
    seed_df, state_df, curves = collect_seed_diagnostics(all_hist, prefix)
    if seed_df.empty:
        continue
    seed_df.round(4).to_csv(OUT_DIR / f"diagnostic_{split_name}_per_seed.csv",
                            index=False)
    state_df.round(4).to_csv(
        OUT_DIR / f"diagnostic_{split_name}_per_state_per_seed.csv", index=False)
 
    g = seed_df.groupby(["mode", "agg"])[["auc", "ap"]]
    mean, std = g.mean(), g.std()          # std: ddof=1
    diag_df = mean.round(4).astype(str) + " ± " + std.round(4).astype(str)
    diag_df["prevalensi"] = (seed_df.groupby(["mode", "agg"])["prevalensi"]
                             .mean().round(4))
    diag_df.index = [f"{m}+{a}" for m, a in diag_df.index]
    diag_df.index.name = "config"
    diag_frames[split_name] = diag_df
    print(f"\nAUC-ROC & AP [{split_name.upper()}], mean ± std antar seed "
          f"(AP acak = prevalensi):\n", diag_df.to_string())
    diag_df.to_csv(OUT_DIR / f"diagnostic_{split_name}_mean_std.csv")
 
    plot_seed_curves(curves, seed_df,
                     OUT_DIR / f"roc_pr_curves_seed_{split_name}.png",
                     title_tail=f"[{split_name.upper()}] (macro antar client, "
                                f"rata-rata seed ± std)")
 
# kurva POOLED (seed pertama) antar agregasi, per mode, val & test
for mode, hs in all_hist.items():
    for split_name, key in (("val", "val_best_curves"), ("test", "test_best_curves")):
        plot_comparison_curves(
            hs, str(OUT_DIR / f"{mode}_roc_pr_pooled_{split_name}.png"),
            split=key)
 
# %%
# ============================================================
# 10. DETAIL KOMPARASI PER CLIENT (val lalu test)
# ============================================================
for mode, hs in all_hist.items():
    print(f"\n######## {mode} : VALIDASI, ronde terbaik, seed pertama ########")
    print_comparison(hs, split="val_best_per_client", baseline="fedavg")
    print(f"\n######## {mode} : TEST, ronde terbaik, seed pertama ########")
    print_comparison(hs, split="test_best_per_client", baseline="fedavg")
    if len(SEEDS) > 1:
        print_seed_summary(hs, split="val_best", ddof=1)
        print_seed_summary(hs, split="test_best", ddof=1)
    for agg, h in hs.items():
        print(f"\n--- {mode}+{agg}: val vs test (seed pertama) ---")
        print_val_test_comparison(h)
 
# ============================================================
# 11. EKSPOR CSV
# ============================================================
for mode, hs in tqdm(all_hist.items(), desc="Ekspor CSV", unit="mode"):
    for agg, h in hs.items():
        tag = f"{mode}_{agg}"
        export_history_csv(h, str(OUT_DIR / f"{tag}_rounds.csv"))
        # validasi: per ronde + akhir (terakhir/terbaik)
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_val_rounds.csv"))
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_val_last.csv"),
                              split="val_last_per_client")
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_val_best.csv"),
                              split="val_best_per_client")
        # test
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_test.csv"),
                              split="test_per_client")
        export_per_client_csv(h, str(OUT_DIR / f"{tag}_per_client_test_best.csv"),
                              split="test_best_per_client")
        # kurva & sweep
        for sp in ("val", "test"):
            export_curves_csv(h, str(OUT_DIR / f"{tag}_curves_{sp}.csv"),
                              split=f"{sp}_best_curves")
            export_threshold_sweep_csv(
                h, str(OUT_DIR / f"{tag}_threshold_sweep_{sp}.csv"),
                split=f"{sp}_best_sweep")
        export_val_test_summary_csv(h, str(OUT_DIR / f"{tag}_val_vs_test.csv"))
    for sp in ("val", "test"):
        export_comparison_csv(
            hs, str(OUT_DIR / f"{mode}_compare_{sp}_best.csv"),
            split=f"{sp}_best_per_client")
        export_seed_summary_csv(hs, str(OUT_DIR / f"{mode}_seed_summary_{sp}.csv"),
                                split=f"{sp}_best", ddof=1)
        export_seed_runs_csv(hs, str(OUT_DIR / f"{mode}_seed_runs_{sp}.csv"),
                             split=f"{sp}_best")
    export_rounds_comparison_csv(hs, str(OUT_DIR / f"{mode}_compare_rounds.csv"))
summary.to_csv(OUT_DIR / "summary.csv")
 
# %%
# ============================================================
# 12. PLOT KURVA F1 PER RONDE (validasi; mean ± std antar seed)
# ============================================================
fig, axes = plt.subplots(1, len(all_hist), figsize=(12, 4), sharey=True)
axes = np.atleast_1d(axes)
for ax, (mode, hs) in zip(axes, all_hist.items()):
    for agg, h in hs.items():
        runs = [r for r in h.get("runs", [h]) if r["f1"]]
        L = min(len(r["f1"]) for r in runs)
        arr = np.array([r["f1"][:L] for r in runs], dtype=float)
        x = runs[0]["round"][:L]
        m, s = np.nanmean(arr, 0), np.nanstd(arr, 0)
        ax.plot(x, m, marker="o", label=agg)
        ax.fill_between(x, m - s, m + s, alpha=.15)
    ax.set_title(f"mode = {mode}")
    ax.set_xlabel("Ronde")
    ax.grid(alpha=.3)
    ax.legend()
axes[0].set_ylabel("Val F1 (rata-rata client, mean ± std seed)")
plt.tight_layout()
plt.savefig(OUT_DIR / "f1_per_round.png", dpi=150)
plt.show()
 
# %%
# ============================================================
# 13. (OPSIONAL) collect_result UNTUK PIPELINE FLOWER LAMA
#     Butuh: fl, config, load_global_model, predict_proba_clients
#     (milik pipeline Flower Anda, bukan fl_runner). Lewati bila tidak dipakai.
# ============================================================
def collect_result(secs):
    model = load_global_model()                           # noqa: F821
    pooled = fl.evaluate_pooled(model, split="test")      # noqa: F821
    df_test = fl.evaluate_on_clients(model, split="test") # noqa: F821
    ronde = pd.read_csv(config.OUT_DIR / "ronde_log.csv").reset_index(drop=True)  # noqa: F821
    bw, _ = fl.bandwidth_report(verbose=False)            # noqa: F821
 
    ys, ps = predict_proba_clients(model, "test")         # noqa: F821
    diag = diagnostic_metrics(ys, ps)
 
    f1 = ronde["f1"].astype(float)
    val_akhir = float(f1.iloc[-1]) if len(f1) else float("nan")
    if f1.notna().any():
        best = f1.idxmax()
        val_best, best_round = float(f1[best]), int(ronde.loc[best, "round"])
    else:
        val_best, best_round = float("nan"), -1
 
    pooled_num = {k: float(v) for k, v in pooled.items() if np.isscalar(v)}
    row = {
        **pooled_num,
        "auc": diag["auc"],
        "ap": diag["ap"],
        "prauc": diag["prauc"],
        "prevalensi": diag["prevalensi"],
        "f1_macro_klien": float(df_test["f1"].mean()),
        "f1_std_klien": float(df_test["f1"].std()),
        "val_f1_akhir": val_akhir,
        "val_f1_terbaik": val_best,
        "ronde_terbaik": best_round,
        "total_MB": float(bw["total_MB"]),
        "waktu_s": round(secs, 1),
    }
    return row, df_test["f1"], ronde, diag

smoke: 100%|██████████| 2/2 [00:22<00:00, 11.36s/ronde, ap=0.414, auc=0.723, f1=0.295, loss=0.507]


val f1/ap per ronde : [0.08233387474567468, 0.2948966201943151] [0.3739655901698829, 0.41417569520355035]
VAL  ronde terbaik  : {'accuracy': 0.779858713881953, 'precision': 0.4875755892508558, 'recall': 0.21379633327182254, 'f1': 0.2948966201943151, 'auc': 0.7227252133962242, 'ap': 0.41417569520355035}
TEST ronde terbaik  : {'accuracy': 0.7882441736651373, 'precision': 0.4795695472192648, 'recall': 0.2190626819158906, 'f1': 0.29740291855812173, 'auc': 0.7204190021375226, 'ap': 0.4023272619136988}
comm                : {'theta_size_mb': 0.0250244140625, 'total_up_mb': 2.2021484375, 'total_down_mb': 2.2021484375, 'total_mb': 4.404296875, 'avg_per_round_mb': 2.2021484375, 'avg_per_client_mb': 0.10009765625, 'full_total_mb': 4.4486083984375, 'saving_pct': 0.9960760639903365}


exp alt+pairwise: 100%|██████████| 10/10 [00:33<00:00,  3.38s/ronde, ap=0.455, auc=0.749, f1=0.447, loss=0.485]


[ronde 10] val F1 0.4468 | AUC 0.7488 | AP 0.4547 | bandwidth ronde 2.202 MB
=== Bandwidth komunikasi (yang benar-benar dikirim: theta) ===
Total downlink (server->client): 11.011 MB
Total uplink   (client->server): 11.011 MB
TOTAL                          : 22.021 MB
Rata-rata per ronde            : 2.202 MB
Rata-rata per client (total)   : 0.500 MB
[Pembanding] bila FULL parameter: 22.243 MB (hemat 1.00%)
 round  val_loss    acc   prec    rec     f1    auc     ap  n_client_fit  down_MB  up_MB  total_MB
     1    0.5190 0.7829 0.5261 0.0767 0.1190 0.7132 0.4024            44   1.1011 1.1011    2.2021
     2    0.4916 0.7751 0.4752 0.3239 0.3812 0.7350 0.4280            44   1.1011 1.1011    2.2021
     3    0.4879 0.7651 0.4555 0.4010 0.4232 0.7424 0.4407            44   1.1011 1.1011    2.2021
     4    0.4859 0.7641 0.4555 0.4215 0.4352 0.7456 0.4469            44   1.1011 1.1011    2.2021
     5    0.4854 0.7643 0.4569 0.4315 0.4414 0.7473 0.4505            44   1.1011 1.1011    2.

Total run: 100%|██████████| 30/30 [51:13<00:00, 102.45s/run, alt+pairwise seed=3407]


...

Ekspor CSV:  50%|█████     | 1/2 [00:00<00:00,  2.36mode/s]

[export] titik kurva ROC/PR tersimpan di: hasil_fl\sim_fedprox_curves_test.csv
[export] threshold sweep tersimpan di: hasil_fl\sim_fedprox_threshold_sweep_test.csv
[export] ringkasan val vs test tersimpan di: hasil_fl\sim_fedprox_val_vs_test.csv
[export] metrik per-ronde tersimpan di: hasil_fl\sim_pairwise_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl\sim_pairwise_per_client_val_rounds.csv
[export] metrik val_last_per_client tersimpan di: hasil_fl\sim_pairwise_per_client_val_last.csv
[export] metrik val_best_per_client tersimpan di: hasil_fl\sim_pairwise_per_client_val_best.csv
[export] metrik test_per_client tersimpan di: hasil_fl\sim_pairwise_per_client_test.csv
[export] metrik test_best_per_client tersimpan di: hasil_fl\sim_pairwise_per_client_test_best.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl\sim_pairwise_curves_val.csv
[export] threshold sweep tersimpan di: hasil_fl\sim_pairwise_threshold_sweep_val.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl

Ekspor CSV: 100%|██████████| 2/2 [00:00<00:00,  2.40mode/s]

[export] threshold sweep tersimpan di: hasil_fl\alt_fedprox_threshold_sweep_test.csv
[export] ringkasan val vs test tersimpan di: hasil_fl\alt_fedprox_val_vs_test.csv
[export] metrik per-ronde tersimpan di: hasil_fl\alt_pairwise_rounds.csv
[export] metrik per-client tersimpan di: hasil_fl\alt_pairwise_per_client_val_rounds.csv
[export] metrik val_last_per_client tersimpan di: hasil_fl\alt_pairwise_per_client_val_last.csv
[export] metrik val_best_per_client tersimpan di: hasil_fl\alt_pairwise_per_client_val_best.csv
[export] metrik test_per_client tersimpan di: hasil_fl\alt_pairwise_per_client_test.csv
[export] metrik test_best_per_client tersimpan di: hasil_fl\alt_pairwise_per_client_test_best.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl\alt_pairwise_curves_val.csv
[export] threshold sweep tersimpan di: hasil_fl\alt_pairwise_threshold_sweep_val.csv
[export] titik kurva ROC/PR tersimpan di: hasil_fl\alt_pairwise_curves_test.csv
[export] threshold sweep tersimpan di: hasil_fl\a


C:\Users\s2\AppData\Local\Temp\ipykernel_8844\958452703.py:335: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
